In [3]:
# ============================================
# PROYECTO: TRANSFORMACIÓN, AGRUPACIÓN Y ANÁLISIS PERSONALIZADO
# Dataset: dataset.csv (ventas de una tienda, Kaggle)
# ============================================

# ============================================
# 0. PREPARACIÓN: cargar y dejar los datos listos
# ============================================
import pandas as pd
from google.colab import files
from IPython.display import display

# Aquí eliges el archivo dataset.csv desde tu computador
archivos = files.upload()
df = pd.read_csv(list(archivos.keys())[0])

# Nombres de columnas en español, cortos y sin espacios (más fácil de escribir)
df = df.rename(columns={
    "Transaction ID": "ID_Transaccion",
    "Date": "Fecha",
    "Customer ID": "ID_Cliente",
    "Gender": "Genero",
    "Age": "Edad",
    "Product Category": "Producto",     # aquí cada categoría hace el papel de "producto"
    "Quantity": "Cantidad",
    "Price per Unit": "Precio_Unitario",
    "Total Amount": "Total",
})

# La fecha viene como texto: la pasamos a fecha real para poder sacar el mes
df["Fecha"] = pd.to_datetime(df["Fecha"])

# OJO: este dataset NO tiene una columna "Tienda".
# Para hacer los cruces que pide el enunciado usaremos Genero y Grupo_Edad
# como segunda columna de agrupación (cumplen el mismo papel: dividir las ventas en grupos).

print("Datos cargados:", df.shape[0], "filas y", df.shape[1], "columnas")
display(df.head())

# ============================================
# 1. TRANSFORMACIÓN DE DATOS
# ============================================

# --------------------------------------------
# 1.1 CREAR NUEVAS COLUMNAS (de lo más sencillo a lo más completo)
# --------------------------------------------

# Ejemplo 1 (sencillo): ingreso por venta = cantidad x precio.
# Lo calculamos nosotros para comprobar que coincide con la columna Total.
df["Ingreso_Venta"] = df["Cantidad"] * df["Precio_Unitario"]
print("\nVentas donde el ingreso calculado no coincide con Total:",
      (df["Ingreso_Venta"] != df["Total"]).sum())

# Ejemplo 2 (intermedio): sacar el mes de la fecha, para poder agrupar por mes después.
df["Mes"] = df["Fecha"].dt.month

# Ejemplo 3 (completo): normalizar las ventas.
# Normalizar = dejar los valores entre 0 y 1 (0 = la venta más baja, 1 = la más alta).
# Sirve para comparar ventas en una misma escala.
minimo = df["Total"].min()
maximo = df["Total"].max()
df["Total_Normalizado"] = ((df["Total"] - minimo) / (maximo - minimo)).round(3)

# --------------------------------------------
# 1.2 CLASIFICAR LAS VENTAS EN CATEGORÍAS (Alta / Media / Baja)
# --------------------------------------------

# Ejemplo 1 (sencillo): clasificar con montos fijos que elegimos nosotros.
# pd.cut corta los valores en rangos: hasta 150 = Baja, de 151 a 600 = Media, más de 600 = Alta
df["Clasificacion_Fija"] = pd.cut(
    df["Total"],
    bins=[0, 150, 600, float("inf")],
    labels=["Baja", "Media", "Alta"],
)

# Ejemplo 2 (intermedio): clasificar en 3 grupos del mismo tamaño.
# pd.qcut reparte las ventas para que cada grupo tenga más o menos la misma cantidad.
df["Clasificacion_Grupos"] = pd.qcut(df["Total"], q=3, labels=["Baja", "Media", "Alta"])

# Ejemplo 3 (completo): una regla propia que mira varias columnas a la vez.
def clasificar_venta(fila):
    if fila["Total"] >= 1000 or (fila["Total"] >= 500 and fila["Cantidad"] >= 3):
        return "Alta"
    elif fila["Total"] >= 150:
        return "Media"
    else:
        return "Baja"

df["Clasificacion_Venta"] = df.apply(clasificar_venta, axis=1)   # axis=1 = fila por fila

print("\n--- Cantidad de ventas por cada tipo de clasificación ---")
print(df["Clasificacion_Fija"].value_counts(), "\n")
print(df["Clasificacion_Grupos"].value_counts(), "\n")
print(df["Clasificacion_Venta"].value_counts())

print("\n--- Datos con las columnas nuevas ---")
display(df[["Producto", "Total", "Ingreso_Venta", "Total_Normalizado", "Clasificacion_Venta"]].head(10))

# Grupos de edad (los usaremos para agrupar, en lugar de "tienda")
df["Grupo_Edad"] = pd.cut(
    df["Edad"],
    bins=[0, 24, 39, 54, 120],
    labels=["Joven (hasta 24)", "Adulto joven (25-39)", "Adulto (40-54)", "Mayor (55+)"],
)

# ============================================
# 2. AGRUPACIÓN Y AGREGACIÓN
# ============================================
# Funciones de agregación que usaremos:
# sum=suma, mean=promedio, count=cantidad, min=mínimo, max=máximo,
# std=desviación estándar (qué tanto varían los valores), var=varianza (la desviación al cuadrado)

# --- Ejemplo 1 (sencillo): agrupar por UNA columna, con todas las funciones ---
print("\n--- Ventas por producto ---")
por_producto = df.groupby("Producto")["Total"].agg(["sum", "mean", "count", "min", "max", "std", "var"]).round(2)
display(por_producto)

# --- Ejemplo 2 (intermedio): agrupar por DOS columnas (producto y género) ---
print("\n--- Ventas por producto y género ---")
por_producto_genero = df.groupby(["Producto", "Genero"])["Total"].agg(
    ["sum", "mean", "count", "min", "max", "std", "var"]
).round(2)
display(por_producto_genero)

# --- Ejemplo 3 (completo): producto y mes, con cuentas distintas por columna ---
print("\n--- Ventas por producto y mes ---")
por_producto_mes = df.groupby(["Producto", "Mes"]).agg(
    Ventas_Totales=("Total", "sum"),
    Venta_Promedio=("Total", "mean"),
    Venta_Minima=("Total", "min"),
    Venta_Maxima=("Total", "max"),
    Desviacion=("Total", "std"),
    Varianza=("Total", "var"),
    Unidades=("Cantidad", "sum"),
    Compras=("Total", "count"),
).round(2)
display(por_producto_mes)

# Vista tipo tabla para comparar a simple vista: cada fila un mes, cada columna un producto
print("\n--- Tabla de ventas totales: meses (filas) y productos (columnas) ---")
display(df.pivot_table(values="Total", index="Mes", columns="Producto", aggfunc="sum"))

# También por producto y grupo de edad
print("\n--- Ventas por producto y grupo de edad ---")
display(df.groupby(["Producto", "Grupo_Edad"], observed=True)["Total"].agg(
    ["sum", "mean", "count", "min", "max", "std", "var"]
).round(2))

# ============================================
# 3. ANÁLISIS PERSONALIZADO CON APPLY
# ============================================

# --- Ejemplo 1 (sencillo): aplicar una función a cada valor de una columna ---
# Pasamos el género al español.
df["Genero_ES"] = df["Genero"].apply(lambda g: "Hombre" if g == "Male" else "Mujer")

# --- Ejemplo 2 (intermedio): aplicar una función propia a cada GRUPO ---
# Algo que sum/mean no calculan solos: qué tan "parejas" son las ventas de un producto.
# Coeficiente de variación = desviación / promedio. Si es bajo, las ventas son parecidas entre sí;
# si es alto, hay mucha diferencia entre una venta y otra.
def variabilidad(ventas):
    return round(ventas.std() / ventas.mean(), 2)

print("\n--- Variabilidad de las ventas por producto (función propia con apply) ---")
display(df.groupby("Producto")["Total"].apply(variabilidad))

# --- Ejemplo 3 (avanzado): desviación de cada venta respecto a la media de su grupo ---
# Para cada venta calculamos: venta - promedio de su grupo (producto y género).
# Resultado positivo = vendió más que el promedio de su grupo; negativo = vendió menos.
df["Desv_vs_Media_Grupo"] = (
    df.groupby(["Producto", "Genero"], group_keys=False)["Total"]
      .apply(lambda ventas: ventas - ventas.mean())
      .round(2)
)

# Lo mismo en porcentaje, para entender "cuánto más o menos" en proporción
df["Desv_Porcentaje"] = (
    df.groupby(["Producto", "Genero"], group_keys=False)["Total"]
      .apply(lambda ventas: (ventas - ventas.mean()) / ventas.mean() * 100)
      .round(1)
)

# Marcamos las ventas que se alejan MUCHO de su grupo (más de 2 desviaciones estándar)
def es_atipica(fila):
    desviacion_grupo = df[(df["Producto"] == fila["Producto"]) & (df["Genero"] == fila["Genero"])]["Total"].std()
    return abs(fila["Desv_vs_Media_Grupo"]) > 2 * desviacion_grupo

df["Venta_Atipica"] = df.apply(es_atipica, axis=1)

print("\n--- Desviación de cada venta respecto a la media de su grupo ---")
display(df[["Producto", "Genero", "Total", "Desv_vs_Media_Grupo", "Desv_Porcentaje", "Venta_Atipica"]].head(10))

print("\nVentas atípicas encontradas:", df["Venta_Atipica"].sum())

# Comprobación: dentro de cada grupo, las desviaciones deberían sumar casi 0
print("\n--- Comprobación (la suma por grupo debe dar ~0) ---")
display(df.groupby(["Producto", "Genero"])["Desv_vs_Media_Grupo"].sum().round(2))

# ============================================
# 4. DOCUMENTACIÓN
# ============================================
# Resumen de lo que se hizo y por qué:
# - Cargamos el archivo, renombramos columnas a español y pasamos la fecha a fecha real.
# - Como no existe columna "Tienda", usamos Género y Grupo de edad como segunda forma de agrupar.
# - Ingreso_Venta (cantidad x precio) se comparó con Total para confirmar que los datos son confiables.
# - Total_Normalizado deja todas las ventas en una escala de 0 a 1 para poder compararlas.
# - Clasificamos las ventas en Alta / Media / Baja de tres maneras (montos fijos, grupos iguales
#   y una regla propia) para ver cómo cambia el resultado según el criterio.
# - Agrupamos por producto, producto y género, producto y mes, y producto y edad, aplicando
#   sum, mean, count, min, max, std y var para describir cada grupo.
# - Con apply hicimos cálculos que las funciones estándar no hacen: la variabilidad de cada
#   producto y la desviación de cada venta respecto al promedio de su grupo.



Saving dataset.csv to dataset.csv
Datos cargados: 1000 filas y 9 columnas


,ID_Transaccion,Fecha,ID_Cliente,Genero,Edad,Producto,Cantidad,Precio_Unitario,Total
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100



Ventas donde el ingreso calculado no coincide con Total: 0

--- Cantidad de ventas por cada tipo de clasificación ---
Clasificacion_Fija
Baja     542
Alta     264
Media    194
Name: count, dtype: int64 

Clasificacion_Grupos
Media    352
Baja     349
Alta     299
Name: count, dtype: int64 

Clasificacion_Venta
Baja     500
Alta     264
Media    236
Name: count, dtype: int64

--- Datos con las columnas nuevas ---


,Producto,Total,Ingreso_Venta,Total_Normalizado,Clasificacion_Venta
0,Beauty,150,150,0.063,Media
1,Clothing,1000,1000,0.494,Alta
2,Electronics,30,30,0.003,Baja
3,Clothing,500,500,0.241,Media
4,Beauty,100,100,0.038,Baja
5,Beauty,30,30,0.003,Baja
6,Clothing,50,50,0.013,Baja
7,Electronics,100,100,0.038,Baja
8,Electronics,600,600,0.291,Media
9,Clothing,200,200,0.089,Media



--- Ventas por producto ---


,sum,mean,count,min,max,std,var
Producto,,,,,,,
Beauty,143515,467.48,307,25,2000,563.61,317659.37
Clothing,155580,443.25,351,25,2000,550.70,303265.99
Electronics,156905,458.79,342,25,2000,567.54,322101.82


... (10 outputs left out)


--- Desviación de cada venta respecto a la media de su grupo ---


,Producto,Genero,Total,Desv_vs_Media_Grupo,Desv_Porcentaje,Venta_Atipica
0,Beauty,Male,150,-337.13,-69.2,False
1,Clothing,Female,1000,532.90,114.1,False
2,Electronics,Male,30,-436.10,-93.6,False
3,Clothing,Male,500,80.20,19.1,False
4,Beauty,Male,100,-387.13,-79.5,False
5,Beauty,Female,30,-420.78,-93.3,False
6,Clothing,Male,50,-369.80,-88.1,False
7,Electronics,Male,100,-366.10,-78.5,False
8,Electronics,Male,600,133.90,28.7,False
9,Clothing,Female,200,-267.10,-57.2,False



Ventas atípicas encontradas: 57

--- Comprobación (la suma por grupo debe dar ~0) ---


Producto     Genero
Beauty       Female    0.52
             Male     -0.33
Clothing     Female   -0.40
             Male      0.40
Electronics  Female    0.40
             Male      0.80
Name: Desv_vs_Media_Grupo, dtype: float64